## Creating the Data Set in DuckDB

This notebook outlines the process of creating a dataset of Indian equities using data from the National Stock Exchange (NSE) and market cap information from Yahoo Finance. The dataset is then used to rank stocks by market capitalization.

### Objectives

1. **Fetch NSE Master List**: Retrieve the list of equities from the NSE website.
2. **Clean Data**: Clean and prepare the data for further processing.
3. **Fetch Market Cap**: Use Yahoo Finance to fetch the latest market cap for each equity.
4. **Rank Stocks**: Rank the stocks by market cap and prepare a final dataset.

### Libraries Used

- **pandas**: For data manipulation and analysis.
- **urllib.request**: For fetching data from the NSE website.
- **yfinance**: For fetching market cap data from Yahoo Finance.
- **duckdb**: For efficient data processing and analysis (not used in this section but planned for later steps).
- **tqdm**: For displaying progress bars during data fetching.

### Data Flow

1. **Fetching NSE Master List**: The NSE master list is fetched from the official NSE website and stored in a pandas DataFrame.
2. **Data Cleaning**: The column names are cleaned, and the data is filtered to include only active equities (those with 'EQ' in the SERIES column).
3. **Market Cap Fetching**: For each equity, the latest market cap is fetched from Yahoo Finance using the `yfinance` library.
4. **Ranking Stocks**: The stocks are ranked by market cap, and a final dataset is prepared.

### Code Overview

The code below demonstrates the steps involved in creating the dataset:

1. **Fetching and Cleaning NSE Data**
2. **Fetching Market Cap Data**
3. **Ranking Stocks by Market Cap**

### Notes

- The code uses a safe throttle (`time.sleep(0.1)`) to avoid hitting rate limits when fetching market cap data.
- The final dataset includes the ticker symbol, company name, and market cap in crores (CR).
- The dataset is sorted in descending order of market cap to identify the largest companies by market capitalization.

### Next Steps

- The next step would be to use **DuckDB** for efficient data processing and analysis of the dataset.
- Further enhancements could include adding more financial metrics, visualizations, and exporting the dataset for use in other applications.

### Conclusion

This notebook provides a comprehensive approach to creating a dataset of Indian equities ranked by market capitalization. The dataset can be used for various financial analyses, including portfolio management, investment research, and market trend analysis.

In [2]:
import pandas as pd
import urllib.request
import io

url = "https://archives.nseindia.com/content/equities/EQUITY_L.csv"
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
response = urllib.request.urlopen(req)
master_df = pd.read_csv(io.StringIO(response.read().decode('utf-8')))

# Strip any leading/trailing whitespace from NSE column names
master_df.columns = master_df.columns.str.strip()

print("Cleaned Columns:", master_df.columns.tolist())
print(master_df[['SYMBOL', 'SERIES', 'PAID UP VALUE', 'FACE VALUE']].head())

Cleaned Columns: ['SYMBOL', 'NAME OF COMPANY', 'SERIES', 'DATE OF LISTING', 'PAID UP VALUE', 'MARKET LOT', 'ISIN NUMBER', 'FACE VALUE']
       SYMBOL SERIES  PAID UP VALUE  FACE VALUE
0   20MICRONS     EQ              5           5
1  21STCENMGM     BE             10          10
2      360ONE     EQ              1           1
3  3BBLACKBIO     EQ             10          10
4   3IINFOLTD     BE             10          10


In [3]:
import yfinance as yf

t = yf.Ticker("RELIANCE.NS")

print("1. Testing fast_info attribute access:")
try:
    print("   t.fast_info.market_cap =", t.fast_info.market_cap)
except Exception as e:
    print("   FAILED:", type(e), e)

print("\n2. Testing fast_info dictionary access:")
try:
    print("   t.fast_info['market_cap'] =", t.fast_info['market_cap'])
except Exception as e:
    print("   FAILED:", type(e), e)

print("\n3. Testing standard .info dictionary:")
try:
    info = t.info
    print("   t.info.get('marketCap') =", info.get('marketCap'))
    print("   Total keys returned in .info:", len(info))
except Exception as e:
    print("   FAILED:", type(e), e)

1. Testing fast_info attribute access:


c:\Users\chira\AppData\Local\Programs\Python\Python314\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\chira\AppData\Local\Programs\Python\Python314\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
c:\Users\chira\AppData\Local\Programs\Python\Python314\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#depreca

   t.fast_info.market_cap = 18071264287161.633

2. Testing fast_info dictionary access:
   t.fast_info['market_cap'] = 18071264287161.633

3. Testing standard .info dictionary:
   t.info.get('marketCap') = 18071263641600
   Total keys returned in .info: 169


In [4]:
import pandas as pd
import urllib.request
import io

# 1. Load NSE Master List
url = "https://archives.nseindia.com/content/equities/EQUITY_L.csv"
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
master_df = pd.read_csv(io.StringIO(urllib.request.urlopen(req).read().decode('utf-8')))

# 2. Clean column names (NSE has leading spaces like ' SERIES')
master_df.columns = master_df.columns.str.strip()

# 3. Filter for active equities ('EQ') and calculate Shares Outstanding
eq_df = master_df[master_df['SERIES'] == 'EQ'].copy()
eq_df['TICKER'] = eq_df['SYMBOL'].astype(str).str.strip() + ".NS"
eq_df['SHARES_OUTSTANDING'] = eq_df['PAID UP VALUE'] / eq_df['FACE VALUE']

print(f"Prepared {len(eq_df)} stocks with exact share counts.")
display(eq_df[['TICKER', 'NAME OF COMPANY', 'PAID UP VALUE', 'FACE VALUE', 'SHARES_OUTSTANDING']].head())

Prepared 2397 stocks with exact share counts.


,TICKER,NAME OF COMPANY,PAID UP VALUE,FACE VALUE,SHARES_OUTSTANDING
0,20MICRONS.NS,20 Microns Limited,5,5,1.0
2,360ONE.NS,360 ONE WAM LIMITED,1,1,1.0
3,3BBLACKBIO.NS,3B Blackbio Dx Limited,10,10,1.0
5,3MINDIA.NS,3M India Limited,10,10,1.0
6,3PLAND.NS,3P Land Holdings Limited,2,2,1.0


In [5]:
import pandas as pd
import yfinance as yf
import duckdb
from datetime import datetime
import time
import warnings
from tqdm import tqdm
import urllib.request
import io

# Suppress Pandas warnings
warnings.filterwarnings("ignore", message=".*inplace keyword.*")
warnings.filterwarnings("ignore")

# 1. Load NSE Master List
url = "https://archives.nseindia.com/content/equities/EQUITY_L.csv"
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
master_df = pd.read_csv(io.StringIO(urllib.request.urlopen(req).read().decode('utf-8')))
master_df.columns = master_df.columns.str.strip()

eq_df = master_df[master_df['SERIES'] == 'EQ'].copy()
eq_df['TICKER'] = eq_df['SYMBOL'].astype(str).str.strip() + ".NS"

all_tickers = eq_df['TICKER'].tolist()
market_caps = {}

print("[*] Fetching live market caps directly to ensure accuracy...")
for i, ticker in enumerate(tqdm(all_tickers, desc="Fetching Market Caps")):
    try:
        # Use fast_info which returned the correct market cap in your test
        t = yf.Ticker(ticker)
        mcap = t.fast_info.get('market_cap', 0)
        if not mcap or mcap == 0:
            # Fallback to info dict if fast_info is missing
            mcap = t.info.get('marketCap', 0)
        
        market_caps[ticker] = mcap
        time.sleep(0.1) # Safe throttle
    except Exception:
        market_caps[ticker] = 0.0

# 2. Map back and rank
eq_df['MARKET_CAP'] = eq_df['TICKER'].map(market_caps).fillna(0.0)
eq_df['MARKET_CAP_CR'] = (eq_df['MARKET_CAP'] / 1e7).round(2)

ranked_universe = eq_df[eq_df['MARKET_CAP'] > 0].sort_values(by='MARKET_CAP', ascending=False).reset_index(drop=True)

print(f"\n[✓] Ranked {len(ranked_universe)} active NSE stocks successfully!")
print(f"    #1 Ranked Stock:   {ranked_universe.loc[0, 'TICKER']} ({ranked_universe.loc[0, 'NAME OF COMPANY']}) - ₹{ranked_universe.loc[0, 'MARKET_CAP_CR']:,.2f} Cr")
print(f"    #500 Cutoff Cap:   {ranked_universe.loc[499, 'TICKER']} ({ranked_universe.loc[499, 'NAME OF COMPANY']}) - ₹{ranked_universe.loc[499, 'MARKET_CAP_CR']:,.2f} Cr")

display(ranked_universe[['TICKER', 'NAME OF COMPANY', 'MARKET_CAP_CR']].head(10))

[*] Fetching live market caps directly to ensure accuracy...


Fetching Market Caps: 100%|██████████| 2397/2397 [23:44<00:00,  1.68it/s]


[✓] Ranked 2345 active NSE stocks successfully!
    #1 Ranked Stock:   RELIANCE.NS (Reliance Industries Limited) - ₹1,807,126.36 Cr
    #500 Cutoff Cap:   PGIL.NS (Pearl Global Industries Limited) - ₹12,857.60 Cr


,TICKER,NAME OF COMPANY,MARKET_CAP_CR
0,RELIANCE.NS,Reliance Industries Limited,1807126.36
1,BHARTIARTL.NS,Bharti Airtel Limited,1316494.40
2,JPL.NS,Jio Platforms Limited,1292487.47
3,HDFCBANK.NS,HDFC Bank Limited,1244768.76
4,ICICIBANK.NS,ICICI Bank Limited,1128542.61
5,SBIN.NS,State Bank of India,979553.07
6,TCS.NS,Tata Consultancy Services Limited,894427.46
7,BAJFINANCE.NS,Bajaj Finance Limited,671434.77
8,LT.NS,Larsen & Toubro Limited,579523.68
9,LICI.NS,Life Insurance Corporation Of India,505999.79


In [6]:
# Change the db path to something more sensible like NSE-universe-raw
db_path = "NSE-universe-raw.duckdb"
conn = duckdb.connect(db_path)
# Add metadata columns to your existing in-memory DataFrame
ranked_universe[' MARKET_CAP_RANK'] = ranked_universe.index + 1
ranked_universe['SYNC_WEEK'] = datetime.now().strftime("%Y-W%V")
ranked_universe['UPDATED_AT'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
# Create persistent Universe Ranking Table and insert directly from Pandas
conn.execute("""
CREATE TABLE IF NOT EXISTS nse_universe_ranks (
ticker VARCHAR PRIMARY KEY,
symbol VARCHAR,
company_name VARCHAR,
isin VARCHAR,
face_value DOUBLE,
market_cap DOUBLE,
market_cap_cr DOUBLE,
market_cap_rank INTEGER,
sync_week VARCHAR,
updated_at VARCHAR
)
""")
conn.execute("""
INSERT OR REPLACE INTO nse_universe_ranks
SELECT 
TICKER,
SYMBOL,
"NAME OF COMPANY",
"ISIN NUMBER",
"FACE VALUE",
MARKET_CAP,
MARKET_CAP_CR,
" MARKET_CAP_RANK",
SYNC_WEEK,
UPDATED_AT
FROM ranked_universe
""")
count = conn.execute("SELECT COUNT(*) FROM nse_universe_ranks").fetchone()[0]
print(f"[✓] Saved {count} ranked NSE equities into 'nse_universe_ranks' in {db_path}!")
conn.close()

[✓] Saved 2345 ranked NSE equities into 'nse_universe_ranks' in NSE-universe-raw.duckdb!


In [3]:
import re
import time
import xml.etree.ElementTree as ET
from collections import Counter
import fitz  # PyMuPDF (pip install pymupdf)
import httpx
import pandas as pd

class NSEFilingTester:
    def __init__(self):
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
            "Accept": "application/json, text/plain, */*",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://www.nseindia.com/companies-listing/corporate-filings-announcements"
        }
        self.client = httpx.Client(headers=self.headers, timeout=25.0, follow_redirects=True)
        self._init_cookies()

    def _init_cookies(self):
        self.client.get("https://www.nseindia.com")
        time.sleep(0.5)

    def discover_filings(self, symbol: str) -> dict:
        ann_url = f"https://www.nseindia.com/api/corporate-announcements?index=equities&symbol={symbol}"
        res = self.client.get(ann_url)
        if res.status_code in (401, 403):
            self._init_cookies()
            res = self.client.get(ann_url)

        found = {
            "symbol": symbol,
            "transcript_url": None,
            "transcript_date": None,
            "presentation_url": None,
            "presentation_date": None,
            "outcome_url": None,
            "xbrl_url": None
        }

        if res.status_code == 200:
            for item in res.json():
                desc = (item.get("desc") or "").lower()
                text = (item.get("attchmntText") or "").lower()
                pdf_url = item.get("attchmntFile")
                dt = item.get("an_dt")
                if not pdf_url or not pdf_url.endswith(".pdf"):
                    continue

                if not found["transcript_url"] and ("transcript" in desc or "transcript" in text or "concall" in text):
                    found["transcript_url"] = pdf_url
                    found["transcript_date"] = dt
                elif not found["presentation_url"] and ("presentation" in desc or "investor presentation" in text):
                    found["presentation_url"] = pdf_url
                    found["presentation_date"] = dt
                elif not found["outcome_url"] and ("outcome of board meeting" in desc or "financial results" in desc):
                    found["outcome_url"] = pdf_url

        # Fetch latest XBRL link from corporate financial results endpoint
        xbrl_api = f"https://www.nseindia.com/api/corporates-financial-results?index=equities&symbol={symbol}&period=Quarterly"
        x_res = self.client.get(xbrl_api)
        if x_res.status_code == 200 and isinstance(x_res.json(), list) and len(x_res.json()) > 0:
            for row in x_res.json():
                xbrl_link = row.get("xbrl")
                if xbrl_link and xbrl_link != "-" and xbrl_link.endswith(".xml"):
                    found["xbrl_url"] = xbrl_link
                    break

        return found

tester = NSEFilingTester()
test_symbol = "DIXON"  # Change to any NSE symbol to test
discovered = tester.discover_filings(test_symbol)
for k, v in discovered.items():
    print(f"{k:20s}: {v}")

symbol              : DIXON
transcript_url      : https://nsearchives.nseindia.com/corporate/DIXON_05112026163605_EarningCallTranscriptQ2Intimation.pdf
transcript_date     : 05-Nov-2026 16:36:21
presentation_url    : https://nsearchives.nseindia.com/corporate/DIXON_30102026161836_OutcomewithPPT30102026.pdf
presentation_date   : 30-Oct-2026 16:19:57
outcome_url         : https://nsearchives.nseindia.com/corporate/DIXON_30102026160547_Outcome30102026.pdf
xbrl_url            : https://nsearchives.nseindia.com/corporate/xbrl/INDAS_117401_1353114_20012025053427.xml


In [4]:
def test_xbrl_and_news(tester: NSEFilingTester, symbol: str, xbrl_url: str):
    # 1. Parse XBRL XML if available
    xbrl_metrics = {}
    if xbrl_url:
        xml_bytes = tester.client.get(xbrl_url).content
        target_tags = {
            "RevenueFromOperations": "Revenue_From_Operations",
            "OtherIncome": "Other_Income",
            "TotalIncome": "Total_Income",
            "CostOfMaterialsConsumed": "Raw_Material_Cost",
            "EmployeeBenefitExpense": "Employee_Benefit_Expense",
            "FinanceCosts": "Finance_Costs",
            "DepreciationDepletionAndAmortisationExpense": "Depreciation",
            "ProfitBeforeTax": "Profit_Before_Tax",
            "ProfitLossForPeriod": "Net_Profit_PAT",
            "BasicEarningsLossPerShareFromContinuingOperations": "Basic_EPS",
            "DateOfStartOfReportingPeriod": "Period_Start",
            "DateOfEndOfReportingPeriod": "Period_End"
        }
        root = ET.fromstring(xml_bytes)
        for elem in root.iter():
            tag = elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag
            if tag in target_tags and elem.text and elem.text.strip():
                val = elem.text.strip()
                try:
                    xbrl_metrics[target_tags[tag]] = float(val)
                except ValueError:
                    xbrl_metrics[target_tags[tag]] = val

    # 2. Fetch Google News RSS
    rss_url = f"https://news.google.com/rss/search?q={symbol}+NSE+stock+India&hl=en-IN&gl=IN&ceid=IN:en"
    rss_res = tester.client.get(rss_url)
    headlines = []
    if rss_res.status_code == 200:
        root = ET.fromstring(rss_res.content)
        for item in root.findall(".//item")[:8]:
            headlines.append(f"[{item.findtext('pubDate', '')[:16]}] {item.findtext('title', '')}")

    return xbrl_metrics, headlines

xbrl_data, news_list = test_xbrl_and_news(tester, test_symbol, discovered["xbrl_url"])
print("=== DETERMINISTIC XBRL QUARTERLY NUMBERS ===")
for k, v in xbrl_data.items():
    print(f"  {k:28s}: {v}")

print("\n=== LATEST NEWS HEADLINES (RAW FACTS) ===")
print("\n".join(news_list))

=== DETERMINISTIC XBRL QUARTERLY NUMBERS ===
  Period_Start                : 2024-04-01
  Period_End                  : 2024-12-31
  Revenue_From_Operations     : 285675600000.0
  Other_Income                : 89500000.0
  Raw_Material_Cost           : 267003100000.0
  Employee_Benefit_Expense    : 4241000000.0
  Finance_Costs               : 1080900000.0
  Depreciation                : 1951100000.0
  Profit_Before_Tax           : 9801500000.0
  Net_Profit_PAT              : 7676300000.0
  Basic_EPS                   : 128.24

=== LATEST NEWS HEADLINES (RAW FACTS) ===
[Thu, 12 Nov 2026] Dixon Technologies (India) Limited (DIXON.NS) Stock Price, News, Quote & History - Yahoo! Finance Canada
[Sun, 08 Nov 2026] We Don’t Think Dixon Technologies (India)'s (NSE:DIXON) Earnings Should Make Shareholders Too Comfortable - Simply Wall St
[Thu, 25 Jun 2026] DIXON Forecast — Price Target — Prediction for 2027 - TradingView
[Mon, 03 Aug 2026] Dixon Tech stock drops 3% post Q1; brokerages see up to

In [9]:
import re
import time
import xml.etree.ElementTree as ET
from collections import Counter
import fitz  # PyMuPDF (pip install pymupdf)
import httpx
import pandas as pd

# ==========================================================
# 1. DETERMINISTIC EXTRACTOR CLASS (IN-MEMORY)
# ==========================================================
class DeterministicFinancialExtractor:
    LEGAL_BOILERPLATE = re.compile(
        r"(safe harbor|forward-looking statements|within the meaning of applicable securities laws|"
        r"actual results may differ materially|no obligation to publicly update|"
        r"for information purposes only|does not constitute an offer)",
        re.IGNORECASE
    )

    PLEASANTRIES = re.compile(
        r"^(thank you.*|good morning.*|good evening.*|congratulations on a (good|great|strong) set.*|"
        r"moderator:.*|operator:.*|ladies and gentlemen.*|i now hand the conference over.*)$",
        re.IGNORECASE | re.MULTILINE
    )

    SIGNAL_KEYWORDS = re.compile(
        r"\b(margin|ebitda|capex|guidance|order book|utilization|capacity|tailwind|headwind|"
        r"pli|subsidy|export|import|duty|tariff|joint venture|jv|acquisition|r&d|market share|"
        r"attrition|raw material|crude|realization|volume growth|bps|crore|cr|fy\d{2})\b",
        re.IGNORECASE
    )

    @staticmethod
    def parse_nse_xbrl_bytes(xml_bytes: bytes) -> dict:
        if not xml_bytes:
            return {}
        target_tags = {
            "RevenueFromOperations": "Revenue_From_Operations",
            "OtherIncome": "Other_Income",
            "TotalIncome": "Total_Income",
            "CostOfMaterialsConsumed": "Raw_Material_Cost",
            "EmployeeBenefitExpense": "Employee_Benefit_Expense",
            "FinanceCosts": "Finance_Costs",
            "DepreciationDepletionAndAmortisationExpense": "Depreciation",
            "ProfitBeforeTax": "Profit_Before_Tax",
            "ProfitLossForPeriod": "Net_Profit_PAT",
            "BasicEarningsLossPerShareFromContinuingOperations": "Basic_EPS",
            "DateOfStartOfReportingPeriod": "Period_Start",
            "DateOfEndOfReportingPeriod": "Period_End"
        }
        extracted = {}
        try:
            root = ET.fromstring(xml_bytes)
            for elem in root.iter():
                clean_tag = elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag
                if clean_tag in target_tags and elem.text and elem.text.strip():
                    val = elem.text.strip()
                    try:
                        extracted[target_tags[clean_tag]] = float(val)
                    except ValueError:
                        extracted[target_tags[clean_tag]] = val
        except Exception:
            pass
        return extracted

    def clean_investor_presentation(self, pdf_bytes: bytes, max_slides: int = 12) -> str:
        if not pdf_bytes:
            return ""
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        raw_slides = []
        line_counts = Counter()

        for page_idx, page in enumerate(doc):
            text = page.get_text("text")
            lines = [line.strip() for line in text.splitlines() if line.strip()]
            for line in set(lines):
                if len(line) < 80:
                    line_counts[line] += 1
            raw_slides.append((page_idx + 1, lines))

        repeat_threshold = max(3, int(len(doc) * 0.3))
        repeating_headers = {l for l, c in line_counts.items() if c >= repeat_threshold}

        scored_slides = []
        for slide_num, lines in raw_slides:
            clean_lines = [l for l in lines if l not in repeating_headers]
            slide_text = "\n".join(clean_lines)
            if len(slide_text) < 100 or self.LEGAL_BOILERPLATE.search(slide_text):
                continue
            kw_hits = len(self.SIGNAL_KEYWORDS.findall(slide_text))
            num_hits = len(re.findall(r"\b\d+(\.\d+)?%\b|\b₹?\s?\d+(,\d+)*(\.\d+)?\s?(cr|crore|bps|mn|bn)\b", slide_text, re.I))
            score = (kw_hits * 2) + num_hits
            if score >= 3:
                scored_slides.append((score, slide_num, slide_text))

        top_slides = sorted(scored_slides, key=lambda x: x[0], reverse=True)[:max_slides]
        top_slides.sort(key=lambda x: x[1])
        return "\n\n".join(f"[Slide {s_num}]\n{txt}" for _, s_num, txt in top_slides)

    def clean_concall_transcript(self, pdf_bytes: bytes, max_chars: int = 14000) -> str:
        if not pdf_bytes:
            return ""
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        pages = []
        for idx, page in enumerate(doc):
            txt = page.get_text("text")
            if idx == 0 and ("management:" in txt.lower() or "moderator:" in txt.lower() or self.LEGAL_BOILERPLATE.search(txt)):
                continue
            pages.append(txt)

        raw_text = "\n".join(pages)
        cleaned = self.PLEASANTRIES.sub("", raw_text)
        cleaned = re.sub(r"\n\s*Page \d+ of \d+\s*\n", "\n", cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)

        paragraphs = [p.strip() for p in cleaned.split("\n\n") if len(p.strip()) > 60]
        high_signal = []
        for p in paragraphs:
            if self.LEGAL_BOILERPLATE.search(p):
                continue
            if len(self.SIGNAL_KEYWORDS.findall(p)) >= 2 or re.search(r"\d+(\.\d+)?%", p):
                high_signal.append(re.sub(r"\s+", " ", p))

        return "\n\n".join(high_signal)[:max_chars]


# ==========================================================
# 2. NSE SESSION & FILING DISCOVERY CLASS
# ==========================================================
class NSEFilingTester:
    def __init__(self):
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
            "Accept": "application/json, text/plain, */*",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://www.nseindia.com/companies-listing/corporate-filings-announcements"
        }
        self.client = httpx.Client(headers=self.headers, timeout=25.0, follow_redirects=True)
        self._init_cookies()

    def _init_cookies(self):
        self.client.get("https://www.nseindia.com")
        time.sleep(0.5)

    def discover_filings(self, symbol: str) -> dict:
        ann_url = f"https://www.nseindia.com/api/corporate-announcements?index=equities&symbol={symbol}"
        res = self.client.get(ann_url)
        if res.status_code in (401, 403):
            self._init_cookies()
            res = self.client.get(ann_url)

        found = {
            "symbol": symbol,
            "transcript_url": None,
            "transcript_date": None,
            "presentation_url": None,
            "presentation_date": None,
            "outcome_url": None,
            "xbrl_url": None
        }

        if res.status_code == 200:
            for item in res.json():
                desc = (item.get("desc") or "").lower()
                text = (item.get("attchmntText") or "").lower()
                pdf_url = item.get("attchmntFile")
                dt = item.get("an_dt")
                if not pdf_url or not pdf_url.endswith(".pdf"):
                    continue

                if not found["transcript_url"] and ("transcript" in desc or "transcript" in text or "concall" in text):
                    found["transcript_url"] = pdf_url
                    found["transcript_date"] = dt
                elif not found["presentation_url"] and ("presentation" in desc or "investor presentation" in text):
                    found["presentation_url"] = pdf_url
                    found["presentation_date"] = dt
                elif not found["outcome_url"] and ("outcome of board meeting" in desc or "financial results" in desc):
                    found["outcome_url"] = pdf_url

        xbrl_api = f"https://www.nseindia.com/api/corporates-financial-results?index=equities&symbol={symbol}&period=Quarterly"
        x_res = self.client.get(xbrl_api)
        if x_res.status_code == 200 and isinstance(x_res.json(), list):
            for row in x_res.json():
                xbrl_link = row.get("xbrl")
                if xbrl_link and xbrl_link != "-" and xbrl_link.endswith(".xml"):
                    found["xbrl_url"] = xbrl_link
                    break

        return found


# ==========================================================
# 3. EXECUTE TEST ON A SINGLE COMPANY
# ==========================================================
test_symbol = "DIXON"  # Change to RELIANCE, TCS, HDFCBANK, etc.
tester = NSEFilingTester()
extractor = DeterministicFinancialExtractor()

print(f"[*] 1. Discovering latest NSE filings for {test_symbol}...")
discovered = tester.discover_filings(test_symbol)
for k, v in discovered.items():
    print(f"    {k:20s}: {v}")

# 2. Extract XBRL & News
print(f"\n[*] 2. Parsing XBRL XML & Google News RSS for {test_symbol}...")
xbrl_data = {}
if discovered["xbrl_url"]:
    xml_bytes = tester.client.get(discovered["xbrl_url"]).content
    xbrl_data = extractor.parse_nse_xbrl_bytes(xml_bytes)

for k, v in xbrl_data.items():
    print(f"    {k:28s}: {v}")

rss_url = f"https://news.google.com/rss/search?q={test_symbol}+NSE+stock+India&hl=en-IN&gl=IN&ceid=IN:en"
rss_res = tester.client.get(rss_url)
headlines = []
if rss_res.status_code == 200:
    root = ET.fromstring(rss_res.content)
    for item in root.findall(".//item")[:6]:
        headlines.append(f"    - [{item.findtext('pubDate', '')[:16]}] {item.findtext('title', '')}")
print("\n    Latest News Headlines:")
print("\n".join(headlines))

# 3. Benchmark PDF Cleaning & Compression
print(f"\n[*] 3. Running Deterministic PDF Cleaning on {test_symbol}...")
audit_rows = []
cleaned_pres = ""
cleaned_trans = ""

if discovered["presentation_url"]:
    pres_bytes = tester.client.get(discovered["presentation_url"]).content
    with fitz.open(stream=pres_bytes, filetype="pdf") as doc:
        raw_pres_text = "\n".join(p.get_text("text") for p in doc)
        total_pages = len(doc)
    cleaned_pres = extractor.clean_investor_presentation(pres_bytes, max_slides=12)
    audit_rows.append({
        "Document": "Investor Presentation",
        "Total Pages": total_pages,
        "Raw Chars": len(raw_pres_text),
        "Cleaned Chars": len(cleaned_pres),
        "Noise Removed (%)": round((1 - len(cleaned_pres) / max(1, len(raw_pres_text))) * 100, 1),
        "Est. LLM Tokens": len(cleaned_pres) // 4
    })

if discovered["transcript_url"] or discovered["outcome_url"]:
    target_pdf_url = discovered["transcript_url"] or discovered["outcome_url"]
    doc_label = "Concall Transcript" if discovered["transcript_url"] else "Board Outcome"
    trans_bytes = tester.client.get(target_pdf_url).content
    with fitz.open(stream=trans_bytes, filetype="pdf") as doc:
        raw_trans_text = "\n".join(p.get_text("text") for p in doc)
        total_pages = len(doc)
    cleaned_trans = extractor.clean_concall_transcript(trans_bytes, max_chars=14000)
    audit_rows.append({
        "Document": doc_label,
        "Total Pages": total_pages,
        "Raw Chars": len(raw_trans_text),
        "Cleaned Chars": len(cleaned_trans),
        "Noise Removed (%)": round((1 - len(cleaned_trans) / max(1, len(raw_trans_text))) * 100, 1),
        "Est. LLM Tokens": len(cleaned_trans) // 4
    })

if audit_rows:
    display(pd.DataFrame(audit_rows))

if cleaned_pres:
    print("\n--- SAMPLE CLEANED PRESENTATION EXCERPT (FIRST 600 CHARS) ---")
    print(cleaned_pres[:600])

if cleaned_trans:
    print("\n--- SAMPLE CLEANED CONCALL/OUTCOME EXCERPT (FIRST 600 CHARS) ---")
    print(cleaned_trans[:600])

[*] 1. Discovering latest NSE filings for DIXON...
    symbol              : DIXON
    transcript_url      : https://nsearchives.nseindia.com/corporate/DIXON_05112026163605_EarningCallTranscriptQ2Intimation.pdf
    transcript_date     : 05-Nov-2026 16:36:21
    presentation_url    : https://nsearchives.nseindia.com/corporate/DIXON_30102026161836_OutcomewithPPT30102026.pdf
    presentation_date   : 30-Oct-2026 16:19:57
    outcome_url         : https://nsearchives.nseindia.com/corporate/DIXON_30102026160547_Outcome30102026.pdf
    xbrl_url            : https://nsearchives.nseindia.com/corporate/xbrl/INDAS_117401_1353114_20012025053427.xml

[*] 2. Parsing XBRL XML & Google News RSS for DIXON...
    Period_Start                : 2024-04-01
    Period_End                  : 2024-12-31
    Revenue_From_Operations     : 285675600000.0
    Other_Income                : 89500000.0
    Raw_Material_Cost           : 267003100000.0
    Employee_Benefit_Expense    : 4241000000.0
    Finance_Costs 

,Document,Total Pages,Raw Chars,Cleaned Chars,Noise Removed (%),Est. LLM Tokens
0,Investor Presentation,30,52494,10153,80.7,2538
1,Concall Transcript,24,64592,14000,78.3,3500



--- SAMPLE CLEANED PRESENTATION EXCERPT (FIRST 600 CHARS) ---
[Slide 16]
Table of Contents
Dixon Technologies | Earnings Presentation Q2 & H1 FY27
1
Cover
Dixon — The brand behind brands
3
Executive Summary — Q2 FY27
Key highlights (Reported)
6
H1 FY27 Performance Overview
Half  Yearly metrics (Adjusted)
8
Consolidated Results — Q2 & H1 FY27
Quarterly & Half Yearly metrics (Adjusted)
9
Segment Performance
Revenue, Capital employed & ROCE
10
Expenditure Analysis
As a % of operating revenues
2
Table of Contents
Sections and navigation
4
Executive Summary — Q2 FY27
Key highlights (Adjusted)
7
Consolidated Results — Q2 & H1 FY27
Quarterly & Half  Yearly metr

--- SAMPLE CLEANED CONCALL/OUTCOME EXCERPT (FIRST 600 CHARS) ---
Atul Lall: I would like to warmly welcome all our stakeholders to discuss our Q2 performance for Financial Year '26-'27. Key highlights for the quarter are as below: Revenues for the quarter ended September 30, 2026 was INR 21,806 crores. EBITDA excluding fair value gai

In [ ]:
import json
import time
import duckdb
import httpx
from pydantic import BaseModel, Field
from typing import List, Literal

# 1. Define the Pass 1 Schema
class SegmentFact(BaseModel):
    segment_name: str
    revenue_or_margin_trend: str
    reported_driver: str

class ExtractedCatalyst(BaseModel):
    event_type: Literal["CAPEX", "PARTNERSHIP_JV", "POLICY_REGULATION", "PRICING_ACTION", "ORDER_BOOK", "RISK_FACTOR"]
    description: str
    timeline: str

class MasterCompanyFacts(BaseModel):
    core_business_model: str
    segment_facts: List[SegmentFact]
    key_catalysts_and_events: List[ExtractedCatalyst]
    raw_materials_and_macro_sensitivities: List[str]
    explicitly_named_competitors: List[str]
    management_numerical_guidance: str
    factual_news_digest: List[str]

# 2. Load cleaned text from DuckDB (read-only so we don't lock the DB)
conn = duckdb.connect("NSE-universe-raw.duckdb", read_only=True)
tables = [t[0] for t in conn.execute("SHOW TABLES").fetchall()]

if "company_intelligence" in tables:
    row = conn.execute("""
        SELECT symbol, company_name, xbrl_numbers_json, latest_news_text, 
               cleaned_presentation_text, cleaned_transcript_text
        FROM company_intelligence
        WHERE length(cleaned_presentation_text) > 100 OR length(cleaned_transcript_text) > 100
        LIMIT 1
    """).fetchone()
else:
    row = None
conn.close()

if row:
    symbol, company_name, xbrl_str, news_txt, pres_txt, trans_txt = row
    print(f"[*] Loaded real pre-cleaned data for {company_name} ({symbol}) from DuckDB.")
else:
    symbol, company_name = "DIXON", "Dixon Technologies (India) Limited"
    xbrl_str = '{"Revenue_From_Operations": 2180600, "Net_Profit_PAT": 29300}'
    news_txt = "[2026-11-10] Dixon Technologies expands smartphone manufacturing capacity in Noida."
    pres_txt = "[Slide 3] Q2 FY27 Revenue grew 85% YoY to INR 21,806 Cr. EBITDA margin stood at 3.8%."
    trans_txt = "Atul Lall: Revenues for Q2 FY27 were INR 21,806 crores. Margins were impacted by expiration of mobile PLI 1 scheme."
    print(f"[*] Using sample payload for {symbol} (no rows in company_intelligence yet).")

prompt = f"""Extract strictly objective corporate facts for {company_name} (NSE: {symbol}) from the pre-cleaned regulatory filings below.
Do NOT invent numbers or add subjective sentiment.

[XBRL DETERMINISTIC NUMBERS]:
{xbrl_str}

[RECENT NEWS HEADLINES]:
{news_txt}

[CLEANED INVESTOR PRESENTATION]:
{pres_txt[:7000]}

[CLEANED CONCALL TRANSCRIPT / OUTCOME]:
{trans_txt[:10000]}
"""

print(f"[*] Prompt length: {len(prompt):,} chars (~{len(prompt)//4:,} tokens). Calling qwen3:14b...")

t0 = time.perf_counter()
res = httpx.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:14b",
        "messages": [{"role": "user", "content": prompt}],
        "format": MasterCompanyFacts.model_json_schema(),
        "think": False,  # Change to "low" if JSON formatting fails
        "stream": False,
        "options": {
            "temperature": 0.0,
            "num_ctx": 8192
        }
    },
    timeout=6000
)
elapsed = time.perf_counter() - t0
res.raise_for_status()
data = res.json()

# 3. Print Performance Telemetry (Ollama returns durations in nanoseconds)
load_s = data.get("load_duration", 0) / 1e9
prompt_eval_s = data.get("prompt_eval_duration", 0) / 1e9
eval_s = data.get("eval_duration", 0) / 1e9
prompt_tokens = data.get("prompt_eval_count", 0)
gen_tokens = data.get("eval_count", 0)

print(f"\n[✓] Completed in {elapsed:.2f}s total!")
print(f"    Model Load Time : {load_s:.2f}s (only happens on first cold call)")
print(f"    Prompt Eval     : {prompt_tokens} tokens in {prompt_eval_s:.2f}s ({prompt_tokens/max(0.01, prompt_eval_s):.1f} tok/s)")
print(f"    JSON Generation : {gen_tokens} tokens in {eval_s:.2f}s ({gen_tokens/max(0.01, eval_s):.1f} tok/s)")

print("\n--- EXTRACTED JSON OUTPUT ---")
parsed = json.loads(data["message"]["content"])
print(json.dumps(parsed, indent=2))

[*] Loaded real pre-cleaned data for Reliance Industries Limited (RELIANCE) from DuckDB.
[*] Prompt length: 18,857 chars (~4,714 tokens). Calling qwen3:14b...


In [1]:
import subprocess, time, httpx

# 1. Check GPU vs CPU memory split in Ollama right now
print("=== OLLAMA PS (GPU vs CPU SPLIT) ===")
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout or "No model loaded yet")

# 2. Stream a tiny 1-line test (NO JSON schema, NO long prompt) to measure raw hardware tok/s
print("=== STREAMING RAW SPEED TEST ===")
t0 = time.perf_counter()
first_token_time = None
tokens = 0

with httpx.stream(
    "POST",
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:14b",
        "messages": [{"role": "user", "content": "Reply with one sentence: What does Reliance Industries do?"}],
        "think": False,
        "stream": True,
        "options": {"num_ctx": 2048}
    },
    timeout=60.0
) as r:
    for line in r.iter_lines():
        if not line:
            continue
        chunk = httpx._models.jsonlib.loads(line)
        if first_token_time is None and chunk.get("message", {}).get("content"):
            first_token_time = time.perf_counter() - t0
        tokens += 1
        print(chunk.get("message", {}).get("content", ""), end="", flush=True)
        if chunk.get("done"):
            eval_s = chunk.get("eval_duration", 1) / 1e9
            eval_cnt = chunk.get("eval_count", 0)
            print(f"\n\n[✓] Time to first token: {first_token_time:.2f}s | Speed: {eval_cnt / max(0.01, eval_s):.1f} tok/s")

print("\n=== OLLAMA PS AFTER LOAD ===")
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)

=== OLLAMA PS (GPU vs CPU SPLIT) ===
NAME    ID    SIZE    PROCESSOR    CONTEXT    RUNNER    UNTIL 

=== STREAMING RAW SPEED TEST ===
Reliance Industries is a multinational conglomerate based in India, primarily engaged in the energy, petrochemicals, retail, and telecommunications sectors.

[✓] Time to first token: 17.89s | Speed: 4.1 tok/s

=== OLLAMA PS AFTER LOAD ===
NAME         ID              SIZE      PROCESSOR          CONTEXT    RUNNER      UNTIL              
qwen3:14b    bdbd181c33f2    9.8 GB    76%/24% CPU/GPU    2048       llamacpp    4 minutes from now    



In [8]:
import json, re, subprocess, time, duckdb, httpx
from pydantic import BaseModel
from typing import List, Literal

# 1. Clear any hung Ollama runner and start fresh
subprocess.run(["taskkill", "/F", "/IM", "ollama.exe", "/T"], capture_output=True)
time.sleep(2)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# 2. Pydantic Schema for Post-Generation Validation
class SegmentFact(BaseModel):
    segment_name: str
    revenue_or_margin_trend: str
    reported_driver: str

class ExtractedCatalyst(BaseModel):
    event_type: Literal["CAPEX", "PARTNERSHIP_JV", "POLICY_REGULATION", "PRICING_ACTION", "ORDER_BOOK", "RISK_FACTOR"]
    description: str
    timeline: str

class MasterCompanyFacts(BaseModel):
    core_business_model: str
    segment_facts: List[SegmentFact]
    key_catalysts_and_events: List[ExtractedCatalyst]
    raw_materials_and_macro_sensitivities: List[str]
    explicitly_named_competitors: List[str]
    management_numerical_guidance: str
    factual_news_digest: List[str]

# 3. Load 1 company's cleaned data from DuckDB (or fallback)
conn = duckdb.connect("NSE-universe-raw.duckdb", read_only=True)
tables = [t[0] for t in conn.execute("SHOW TABLES").fetchall()]
row = None
if "company_intelligence" in tables:
    row = conn.execute("""
        SELECT symbol, company_name, xbrl_numbers_json, latest_news_text, 
               cleaned_presentation_text, cleaned_transcript_text
        FROM company_intelligence
        WHERE length(cleaned_presentation_text) > 100 OR length(cleaned_transcript_text) > 100
        LIMIT 1
    """).fetchone()
conn.close()

if row:
    symbol, company_name, xbrl_str, news_txt, pres_txt, trans_txt = row
else:
    symbol, company_name = "DIXON", "Dixon Technologies (India) Limited"
    xbrl_str = '{"Revenue_From_Operations": 2180600, "Net_Profit_PAT": 29300}'
    news_txt = "[2026-11-10] Dixon Technologies expands smartphone manufacturing capacity in Noida."
    pres_txt = "[Slide 3] Q2 FY27 Revenue grew 85% YoY to INR 21,806 Cr. EBITDA margin stood at 3.8%."
    trans_txt = "Atul Lall: Revenues for Q2 FY27 were INR 21,806 crores. Margins were impacted by expiration of mobile PLI 1 scheme."

prompt = f"""Extract strictly objective corporate facts for {company_name} (NSE: {symbol}) from the filings below.
Return ONLY a valid JSON object matching this exact structure (no markdown, no commentary):
{{
  "core_business_model": "string",
  "segment_facts": [{{"segment_name": "string", "revenue_or_margin_trend": "string", "reported_driver": "string"}}],
  "key_catalysts_and_events": [{{"event_type": "CAPEX|PARTNERSHIP_JV|POLICY_REGULATION|PRICING_ACTION|ORDER_BOOK|RISK_FACTOR", "description": "string", "timeline": "string"}}],
  "raw_materials_and_macro_sensitivities": ["string"],
  "explicitly_named_competitors": ["string"],
  "management_numerical_guidance": "string",
  "factual_news_digest": ["string"]
}}

[XBRL NUMBERS]: {xbrl_str}
[NEWS]: {news_txt}
[PRESENTATION]: {pres_txt[:6000]}
[CONCALL]: {trans_txt[:9000]}
"""

MODEL_TAG = "nemotron-3-nano:4b"
print(f"[*] Calling {MODEL_TAG} for {symbol} ({len(prompt):,} chars)...")
t0 = time.perf_counter()

res = httpx.post(
    "http://localhost:11434/api/chat",
    json={
        "model": MODEL_TAG,
        "messages": [{"role": "user", "content": prompt}],
        "format": "json",
        "stream": False,
        "options": {
            "temperature": 0.0,
            "num_ctx": 8192,
            "num_predict": 1800  # Leaves room for Nemotron's brief reasoning + full JSON
        }
    },
    timeout=180.0
)
res.raise_for_status()
data = res.json()
elapsed = time.perf_counter() - t0

# Clean any accidental markdown fences or <think> tags before Pydantic validation
raw_text = data["message"]["content"].strip()
raw_text = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL).strip()
raw_text = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text, flags=re.IGNORECASE).strip()

validated = MasterCompanyFacts.model_validate_json(raw_text)
eval_s = data.get("eval_duration", 1) / 1e9
eval_cnt = data.get("eval_count", 0)

print(f"[✓] Success in {elapsed:.2f}s! ({eval_cnt} tokens @ {eval_cnt / max(0.01, eval_s):.1f} tok/s)")
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)
print(validated.model_dump_json(indent=2))

[*] Calling nemotron-3-nano:4b for RELIANCE (17,306 chars)...
[✓] Success in 96.47s! (1502 tokens @ 26.9 tok/s)
NAME                  ID              SIZE      PROCESSOR          CONTEXT    RUNNER      UNTIL              
nemotron-3-nano:4b    d94b6dfae26d    3.3 GB    28%/72% CPU/GPU    8192       llamacpp    4 minutes from now    

{
  "core_business_model": "Diversified conglomerate with strong O2C (Oil & Gas, Chemicals, Digital Services), Retail, Media, New Energy, and Jio Platforms.",
  "segment_facts": [
    {
      "segment_name": "O2C",
      "revenue_or_margin_trend": "Revenue up 44% YoY, EBITDA up 68% YoY",
      "reported_driver": "Tight fuel markets, advantaged feedstock sourcing"
    },
    {
      "segment_name": "Digital Services",
      "revenue_or_margin_trend": "Revenue up 11% YoY, EBITDA up 23.9% YoY",
      "reported_driver": "Higher CBM volumes and condensate realization"
    },
    {
      "segment_name": "Retail",
      "revenue_or_margin_trend": "Revenue down 10

In [9]:
import subprocess

print("=== OLLAMA PS (GPU vs CPU SPLIT) ===")
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)

print("=== GPU HARDWARE CHECK ===")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], capture_output=True, text=True).stdout or "No NVIDIA GPU detected")

=== OLLAMA PS (GPU vs CPU SPLIT) ===
NAME                  ID              SIZE      PROCESSOR          CONTEXT    RUNNER      UNTIL              
nemotron-3-nano:4b    d94b6dfae26d    3.3 GB    28%/72% CPU/GPU    8192       llamacpp    3 minutes from now    

=== GPU HARDWARE CHECK ===
name, memory.total [MiB], memory.free [MiB]
NVIDIA GeForce RTX 3050 Laptop GPU, 4096 MiB, 1544 MiB

